In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
WORKDIR.mkdir(parents=True, exist_ok=True)

repo = "https://github.com/hanswalker/r2dreamer.git"
branch = "main"
if R2DREAMER_DIR.exists():
    run(["git", "remote", "set-url", "origin", repo], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "origin"], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", branch, f"origin/{branch}"], cwd=R2DREAMER_DIR)
    run(["git", "reset", "--hard", f"origin/{branch}"], cwd=R2DREAMER_DIR)
else:
    run(["git", "clone", "--branch", branch, repo, str(R2DREAMER_DIR)])

os.chdir(R2DREAMER_DIR)
sys.path.insert(0, str(R2DREAMER_DIR))

from dmc_expert_colab import setup_colab

TDMPC2_DIR, DATA_DIR, Mamba3 = setup_colab(WORKDIR, R2DREAMER_DIR)


In [ ]:
from dmc_expert_collection import load_collection_config, prepare_collection_run, read_progress, start_collection_run
from IPython.display import clear_output
import time

cfg = load_collection_config(
    R2DREAMER_DIR / "configs" / "dmc_expert_collection.yaml",
    tdmpc2_dir=TDMPC2_DIR,
    data_dir=DATA_DIR,
)

def short(path):
    return str(path.relative_to(DATA_DIR))

jobs = []
for task in cfg["tasks"]:
    run_cfg = prepare_collection_run(cfg, task, r2dreamer_dir=R2DREAMER_DIR)
    proc, log_file = start_collection_run(run_cfg)
    jobs.append((run_cfg, proc, log_file))

while True:
    clear_output(wait=True)
    print(f"Data collection: {len(jobs)} tasks x {cfg['num_episodes']} episodes")
    print(f"{'task':<22} {'status':<10} {'episodes':>11} {'rows':>10}  log")
    done = True

    for run_cfg, proc, _ in jobs:
        code = proc.poll()
        status = "running" if code is None else ("done" if code == 0 else f"failed {code}")
        episodes, rows, _ = read_progress(run_cfg["out_dir"], run_cfg["task"])
        print(f"{run_cfg['task']:<22} {status:<10} {episodes:>4}/{int(cfg['num_episodes']):<6} {rows:>10}  {short(run_cfg['log_path'])}")
        done = done and code is not None

    if done:
        break
    time.sleep(int(cfg["refresh_seconds"]))

failed = []
for run_cfg, proc, log_file in jobs:
    log_file.close()
    if proc.returncode != 0:
        failed.append((run_cfg["task"], run_cfg["log_path"]))

if failed:
    for task, log_path in failed:
        print()
        print(f"--- {task} last log lines ---")
        print(chr(10).join(log_path.read_text(errors="replace").splitlines()[-80:]))
    raise RuntimeError(f"{len(failed)} collectors failed")

print("Collection finished.")


In [ ]:
from dmc_expert_training import default_training_runs, read_metric_summary, start_training_run
from IPython.display import clear_output
import time

TRAIN_STORE = DATA_DIR / "walker_walk" / "walker_walk.zarr"
TRAIN_RUNS = default_training_runs(WORKDIR)

def short(path):
    return str(path.relative_to(WORKDIR))

jobs = []
for run_cfg in TRAIN_RUNS:
    proc, log_file, stdout_log = start_training_run(
        run_cfg,
        r2dreamer_dir=R2DREAMER_DIR,
        train_store=TRAIN_STORE,
        env_task="dmc_walker_walk",
        resume=False,
    )
    jobs.append((run_cfg, proc, log_file, stdout_log))

while True:
    clear_output(wait=True)
    print("Offline training: GRU vs Mamba3")
    print(f"{'model':<8} {'status':<10} {'step':>8} {'loss':>10} {'eval':>10} {'ckpt':<9} log")
    done = True

    for run_cfg, proc, _, stdout_log in jobs:
        code = proc.poll()
        status = "running" if code is None else ("done" if code == 0 else f"failed {code}")
        step, loss, score = read_metric_summary(run_cfg["logdir"])
        ckpt = "latest" if (run_cfg["logdir"] / "latest.pt").exists() else "-"
        ckpt = "best" if (run_cfg["logdir"] / "best.pt").exists() else ckpt
        print(f"{run_cfg['name']:<8} {status:<10} {step:>8} {loss:>10} {score:>10} {ckpt:<9} {short(stdout_log)}")
        done = done and code is not None

    if done:
        break
    time.sleep(10)

failed = []
for run_cfg, proc, log_file, stdout_log in jobs:
    log_file.close()
    if proc.returncode != 0:
        failed.append((run_cfg["name"], stdout_log))

if failed:
    for name, stdout_log in failed:
        print()
        print(f"--- {name} last log lines ---")
        print(chr(10).join(stdout_log.read_text(errors="replace").splitlines()[-80:]))
    raise RuntimeError(f"{len(failed)} training runs failed")

print("Training finished.")


In [ ]:
from dmc_expert_evaluation import default_evaluation_runs, evaluate_training_run

runs = TRAIN_RUNS if "TRAIN_RUNS" in globals() else default_evaluation_runs(WORKDIR)
results = []

for run_cfg in runs:
    print(f"Evaluating {run_cfg['name']}...")
    results.append(
        evaluate_training_run(
            run_cfg,
            r2dreamer_dir=R2DREAMER_DIR,
            train_store=DATA_DIR / "walker_walk" / "walker_walk.zarr",
            env_task="dmc_walker_walk",
            eval_episodes=5,
            checkpoint_name="best.pt",
        )
    )

print()
print("Eval summary")
for item in results:
    print(
        f"{item['model']:<8} fresh={item['fresh_eval_score']} "
        f"best_logged={item['logged_best_eval']} ckpt={item['checkpoint']}"
    )

results
